In [1]:
import pandas as pd
import numpy as np
import hashlib
import os
import warnings
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr, wilcoxon as wilcoxon_test
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import copy
warnings.filterwarnings('ignore')

### Path Configs

In [2]:
PARQUET_PATH = "../data_logs/processed/clean_dataset.parquet"
SURVEY_PATH  = "../data_logs/processed/survey_data.csv"
OUTPUT_DIR   = "../data_logs/processed/"

In [3]:
# features that need normalization
ANATOMICAL_FEATURES   = ['EAR', 'Mouth_W', 'Lip_Gap']
GAZE_FEATURES         = ['Gaze_H', 'Gaze_V']          # Regime A — calibration min-max
HEAD_POSE_FEATURES    = ['Pitch', 'Yaw', 'Roll']       # baseline IQR
REGIME_B_FEATURES     = ['Iris_X', 'Iris_Y']           # baseline IQR
BASELINE_IQR_FEATURES = ANATOMICAL_FEATURES + HEAD_POSE_FEATURES + REGIME_B_FEATURES
SPATIAL_FEATURES = ANATOMICAL_FEATURES + GAZE_FEATURES + HEAD_POSE_FEATURES + REGIME_B_FEATURES

# hard coded transition window with exclusion bounds for the reflex buffer using true_time_in_state 
SEGMENTS = {
    "task1":   {"state": 4, "t_start": 6.0,  "t_end": 15.0, "label": 0, "name": "Task1_Low"},
    "task2_A": {"state": 7, "t_start": 1.0,  "t_end": 10.0, "label": 1, "name": "Task2A_High"},
    "task2_B": {"state": 7, "t_start": 13.5, "t_end": 20.0, "label": 1, "name": "Task2B_High"}
}

# baseline duration hard-coded according to duration set in pygame_app.py
BASELINE = {"state": 2, "t_start": 5.0, "t_end": 20.0}
CALIBRATION = {"state": 1}  # corner calibration

ZERO_GUARD_THRESH = 1e-6
BASELINE_MIN_FRAMES = 150  # atleast 5s of valid data from baseline state 

## Multi-scale window Configs

In [4]:
WINDOW_SCALES = {
    "2s": {"duration": 2.0, "expected_frames": 60,  "M": 30, "grid_G": 5},
    "3s": {"duration": 3.0, "expected_frames": 90,  "M": 45, "grid_G": 6},
    "4s": {"duration": 4.0, "expected_frames": 120, "M": 60, "grid_G": 7},
}
K_DRAWS = 15

`grid_G` is the number of bins per axis in the SGE entropy grid with its values being calculated using &radic;M

In [5]:
REGIME_A_NORM_COLS = ['Gaze_H_norm', 'Gaze_V_norm']
REGIME_B_NORM_COLS = ['Iris_X_norm', 'Iris_Y_norm']
POSE_NORM_COLS     = ['Pitch_norm', 'Yaw_norm', 'Roll_norm']
ANATOMICAL_FEATURES_COL  = ['EAR_norm', 'Mouth_W_norm', 'Lip_Gap_norm']

COLLINEARITY_RHO_THRESH = 0.7
DELTA_TARGET_COLS       = ['A_mpd', 'B_mpd', 'A_sge_mm', 'B_sge_mm']

## Loading and viewing dataset

In [6]:
data_raw = pd.read_parquet(PARQUET_PATH)
df = copy.deepcopy(data_raw)

df.head(5)

,Tracking_Flag,Pitch,Yaw,Roll,EAR,Gaze_H,Gaze_V,Mouth_W,Lip_Gap,Left_Iris_X,...,task2_answer,task2_error,task2_backspaces,memory_code_target,memory_code_answer,memory_code_error,memory_code_backspaces,Session_ID,Relative_Time_Sec,true_time_in_state
0,1.0,-45.665562,37.414612,-0.856360,0.249795,0.5228,0.5970,0.5603,0.0161,-21.843124,...,,-1.0,0.0,,,-1.0,0.0,p18,0.000000,0.000000
1,1.0,-44.882950,37.621552,-0.789102,0.247039,0.5251,0.5922,0.5575,0.0180,-22.011524,...,,-1.0,0.0,,,-1.0,0.0,p18,0.034601,0.034601
2,1.0,-44.982403,37.192730,-0.840032,0.251813,0.5185,0.5806,0.5514,0.0179,-22.342602,...,,-1.0,0.0,,,-1.0,0.0,p18,0.069201,0.069201
3,1.0,-44.722378,36.828007,-0.438024,0.248633,0.5175,0.5740,0.5527,0.0134,-22.563278,...,,-1.0,0.0,,,-1.0,0.0,p18,0.103802,0.103802
4,1.0,-44.749332,36.331085,-0.175422,0.256811,0.5185,0.5779,0.5533,0.0140,-22.913239,...,,-1.0,0.0,,,-1.0,0.0,p18,0.138403,0.138403


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50798 entries, 0 to 50797
Data columns (total 34 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Tracking_Flag           50798 non-null  float32
 1   Pitch                   50798 non-null  float32
 2   Yaw                     50798 non-null  float32
 3   Roll                    50798 non-null  float32
 4   EAR                     50798 non-null  float32
 5   Gaze_H                  50798 non-null  float32
 6   Gaze_V                  50798 non-null  float32
 7   Mouth_W                 50798 non-null  float32
 8   Lip_Gap                 50798 non-null  float32
 9   Left_Iris_X             50798 non-null  float32
 10  Left_Iris_Y             50798 non-null  float32
 11  Right_Iris_X            50798 non-null  float32
 12  Right_Iris_Y            50798 non-null  float32
 13  timestamp               50798 non-null  float64
 14  time_in_state           50798 non-null  float64
 

In [8]:
print(df.columns.tolist())

['Tracking_Flag', 'Pitch', 'Yaw', 'Roll', 'EAR', 'Gaze_H', 'Gaze_V', 'Mouth_W', 'Lip_Gap', 'Left_Iris_X', 'Left_Iris_Y', 'Right_Iris_X', 'Right_Iris_Y', 'timestamp', 'time_in_state', 'FSM_State', 'calibration_flag', 'active_task', 'target_character', 'task1_ground_truth', 'task1_answer', 'task1_error', 'task1_backspaces', 'task2_ground_truth', 'task2_answer', 'task2_error', 'task2_backspaces', 'memory_code_target', 'memory_code_answer', 'memory_code_error', 'memory_code_backspaces', 'Session_ID', 'Relative_Time_Sec', 'true_time_in_state']


In [9]:
# adding Iris_X and Iris_Y in the code to check relatively with gaze ratios to see which gives a better model eval
if 'Iris_X' not in df.columns:
    df['Iris_X'] = (df['Left_Iris_X'] + df['Right_Iris_X']) / 2
if 'Iris_Y' not in df.columns:
    df['Iris_Y'] = (df['Left_Iris_Y'] + df['Right_Iris_Y']) / 2

# Flag hardware failures AND NaN/Inf mathematical corruptions
df['is_valid'] = ((df['Tracking_Flag'] == 1.0) & 
                 (~df[SPATIAL_FEATURES].isna().any(axis=1)) & 
                 (~df[SPATIAL_FEATURES].isin([np.inf, -np.inf]).any(axis=1)))

print(f"Initial Data Shape: {df.shape}. Valid Frames: {df['is_valid'].sum()}")

Initial Data Shape: (50798, 37). Valid Frames: 50636


### Bounded Data Extraction

Extracting select required data from the dataset using the timing bounds we setup earlier which also accounts for reflex buffer of 1s at the start of each task 

In [10]:
def get_segment_mask(df, state, t_start, t_end):
    """Generates a boolean mask strictly bounded by floating-point timestamps."""
    return (
        (df['FSM_State'] == state) &
        (df['is_valid']) &
        (df['true_time_in_state'] >= t_start) &
        (df['true_time_in_state'] <  t_end)
    )

mask_base = get_segment_mask(df, BASELINE["state"], BASELINE["t_start"], BASELINE["t_end"])
baseline_frames = df[mask_base].copy()
baseline_frames.tail(5)

,Tracking_Flag,Pitch,Yaw,Roll,EAR,Gaze_H,Gaze_V,Mouth_W,Lip_Gap,Left_Iris_X,...,memory_code_target,memory_code_answer,memory_code_error,memory_code_backspaces,Session_ID,Relative_Time_Sec,true_time_in_state,Iris_X,Iris_Y,is_valid
49620,1.0,-38.316013,-34.161301,3.357820,0.255395,0.5186,0.5744,0.5493,0.0115,-17.888615,...,,,-1.0,0.0,p17,31.687068,19.762849,3.689988,-27.276451,True
49621,1.0,-38.586220,-34.652020,3.329936,0.258369,0.5231,0.5888,0.5607,0.0108,-17.517885,...,,,-1.0,0.0,p17,31.734575,19.810356,3.652288,-26.961399,True
49622,1.0,-38.738647,-34.098145,3.400327,0.256211,0.5257,0.5956,0.5463,0.0085,-17.562529,...,,,-1.0,0.0,p17,31.782082,19.857863,3.850281,-27.236099,True
49623,1.0,-38.657925,-34.825272,3.517753,0.251029,0.5177,0.5883,0.5524,0.0082,-17.478527,...,,,-1.0,0.0,p17,31.829589,19.905370,3.583693,-27.128576,True
49624,1.0,-38.733696,-34.016857,3.465033,0.255711,0.5194,0.5774,0.5535,0.0090,-17.819740,...,,,-1.0,0.0,p17,31.877096,19.952877,3.653784,-27.339527,True


In [11]:
# Calibration specifically requires valid corner flags
calib_flags = ['TOP_LEFT', 'TOP_RIGHT', 'BOTTOM_LEFT', 'BOTTOM_RIGHT']
mask_calib = (df["FSM_State"] == CALIBRATION["state"]) & (df["is_valid"]) & (df["calibration_flag"].isin(calib_flags))
calib_frames = df[mask_calib].copy()

In [12]:
# active task segments
segment_parts = []
for seg_id, params in SEGMENTS.items():
    mask = get_segment_mask(df, params["state"], params["t_start"], params["t_end"])
    temp_df = df[mask].copy()
    temp_df["class_label"] = params["label"]
    temp_df["segment_name"] = params["name"]
    segment_parts.append(temp_df)

task_frames = pd.concat(segment_parts, ignore_index=True)
task_frames.head(5)

,Tracking_Flag,Pitch,Yaw,Roll,EAR,Gaze_H,Gaze_V,Mouth_W,Lip_Gap,Left_Iris_X,...,memory_code_error,memory_code_backspaces,Session_ID,Relative_Time_Sec,true_time_in_state,Iris_X,Iris_Y,is_valid,class_label,segment_name
0,1.0,-49.699295,42.529781,-1.668356,0.315869,0.5238,0.6102,0.5328,0.0296,-15.655897,...,-1.0,0.0,p18,48.959932,6.020515,0.637416,-23.413528,True,0,Task1_Low
1,1.0,-49.872616,42.999947,-1.804441,0.312996,0.5323,0.5992,0.5344,0.0291,-15.231106,...,-1.0,0.0,p18,48.994532,6.055115,0.741593,-23.636421,True,0,Task1_Low
2,1.0,-49.810848,43.191643,-1.760916,0.315065,0.5295,0.5995,0.5329,0.0295,-15.092757,...,-1.0,0.0,p18,49.029133,6.089716,0.703508,-23.600027,True,0,Task1_Low
3,1.0,-49.640259,43.353054,-1.755763,0.319846,0.5270,0.6028,0.5341,0.0277,-14.987332,...,-1.0,0.0,p18,49.063734,6.124317,0.676390,-23.531994,True,0,Task1_Low
4,1.0,-49.542099,43.453159,-1.640138,0.320057,0.5281,0.6045,0.5373,0.0279,-15.046221,...,-1.0,0.0,p18,49.098334,6.158917,0.586570,-23.598536,True,0,Task1_Low


In [13]:
task_frames.tail(3)

,Tracking_Flag,Pitch,Yaw,Roll,EAR,Gaze_H,Gaze_V,Mouth_W,Lip_Gap,Left_Iris_X,...,memory_code_error,memory_code_backspaces,Session_ID,Relative_Time_Sec,true_time_in_state,Iris_X,Iris_Y,is_valid,class_label,segment_name
12269,1.0,-40.726875,-35.742153,2.883801,0.257550,0.5139,0.6146,0.5283,0.0140,-18.617994,...,-1.0,0.0,p17,79.146411,19.857863,2.401404,-26.095837,True,1,Task2B_High
12270,1.0,-41.072323,-35.925297,3.039200,0.254129,0.5166,0.6277,0.5412,0.0135,-18.597723,...,-1.0,0.0,p17,79.193918,19.905370,2.243582,-25.916029,True,1,Task2B_High
12271,1.0,-40.874847,-34.820480,2.758710,0.250306,0.5204,0.6273,0.5298,0.0127,-18.896534,...,-1.0,0.0,p17,79.241425,19.952877,2.580835,-26.125820,True,1,Task2B_High


In [14]:
print(f"\nBaseline frames  : {len(baseline_frames):,}")
print(f"Calibration frames: {len(calib_frames):,}")
print(f"Task frames      : {len(task_frames):,}")


Baseline frames  : 7,512
Calibration frames: 4,011
Task frames      : 12,272


In [15]:
# To prove exact floating-point bounding
print("\nTemporal Bounding Verification (First 3 Subjects):")
for subj in task_frames['Session_ID'].unique()[:3]:
    for seg_name in task_frames['segment_name'].unique():
        subset = task_frames[(task_frames['Session_ID'] == subj) & (task_frames['segment_name'] == seg_name)]
        if not subset.empty:
            actual_start = subset['true_time_in_state'].min()
            actual_end = subset['true_time_in_state'].max()
            print(f"  {subj} -> {seg_name}: Captured [{actual_start:.3f}s - {actual_end:.3f}s] | {len(subset)} frames")


Temporal Bounding Verification (First 3 Subjects):
  p18 -> Task1_Low: Captured [6.021s - 14.982s] | 260 frames
  p18 -> Task2A_High: Captured [1.003s - 10.000s] | 261 frames
  p18 -> Task2B_High: Captured [13.529s - 19.965s] | 187 frames
  p8 -> Task1_Low: Captured [6.001s - 14.969s] | 267 frames
  p8 -> Task2A_High: Captured [1.011s - 9.980s] | 267 frames
  p8 -> Task2B_High: Captured [13.520s - 19.993s] | 193 frames
  p4 -> Task1_Low: Captured [6.003s - 14.961s] | 192 frames
  p4 -> Task2A_High: Captured [1.032s - 9.990s] | 192 frames
  p4 -> Task2B_High: Captured [13.508s - 19.980s] | 139 frames


We implemented strict floating-point inequalities, as empirically verified by our diagnostic logs showing precise millisecond captures

## Data Normalization

### Subject quality

In [16]:
# identifying users lacking required baseline or calibration flag
subjects_with_enough_baseline = (
    baseline_frames.groupby('Session_ID')
    .size()
    .pipe(lambda s: s[s >= BASELINE_MIN_FRAMES])
    .index
)

subjects_with_calibration = calib_frames['Session_ID'].unique()
subjects_with_calibration

<ArrowStringArray>
['p18',  'p8',  'p4',  'p5',  'p3',  'p9', 'p12', 'p13',  'p2',  'p1',  'p7',
  'p6', 'p14', 'p16', 'p11', 'p10', 'p15', 'p17']
Length: 18, dtype: str

In [17]:
valid_subjects  = set(subjects_with_enough_baseline) & set(subjects_with_calibration)
dropped_subjects = set(task_frames['Session_ID'].unique()) - valid_subjects

print(f"Subjects passing quality gate : {len(valid_subjects)}")
print(f"Subjects dropped : {dropped_subjects}")

Subjects passing quality gate : 18
Subjects dropped : set()


In [18]:
# only using subjects having the required calibration and baseline data
baseline_frames = baseline_frames[baseline_frames['Session_ID'].isin(valid_subjects)].copy()
calib_frames    = calib_frames[calib_frames['Session_ID'].isin(valid_subjects)].copy()
task_frames     = task_frames[task_frames['Session_ID'].isin(valid_subjects)].copy()

### Screen-Boundary Min-Max Spatial Normalization

Due to high intra-subject variance in seated posture, camera-to-face distance, and intrinsic anatomical eye dimensions, raw spatial displacement values (such as `Iris_X/Y` and `Gaze_H/V`) lack standardized analytical utility across a multi-subject cohort. A 2mm positional shift on a 1080p sensor may correspond to a massive screen traversal for one subject, but a minor saccade for another.

To resolve this, we utilized a dedicated structural calibration phase where spatial anchors were collected at the absolute extrema of the visual environment (the four corners of the display). These anchor points establish a personalized, scale-invariant spatial bounding box for each user. 

All subsequent gaze geometries captured during the cognitive load tasks were transformed into a uniform spatial percentage mapping ($[0, 1]$ coordinate space) using a zero-guarded Min-Max normalization protocol. For a given active task frame, the normalized coordinate $x'_{i,f}$ for subject $i$ and feature $f$ is computed as:

$$x'_{i,f} = \frac{x_{i,f} - \min(X_{c, i, f})}{\max\Big(\big(\max(X_{c, i, f}) - \min(X_{c, i, f})\big), \ \epsilon\Big)}$$

Where:
* $x_{i,f}$ is the raw active spatial coordinate.
* $X_{c, i, f}$ is the array of coordinates captured during the subject's screen-corner calibration phase.
* $\epsilon$ is the zero-guard threshold ($\approx 1 \times 10^{-6}$) applied to the denominator to prevent undefined spatial matrices in the event of hardware frame-freezing during calibration.

**Out-of-Bounds Diagnostic Tracking:** By mathematically constraining the coordinate plane strictly to the physical dimensions of the monitor, this protocol allows for empirical distraction tracking. Any active normalized value where $x'_{i,f} \notin [0, 1]$ functions as a mathematical proof of an off-screen saccade (e.g., glancing at a keyboard or environmental distraction), allowing these frames to be systematically logged and quantified.

In [19]:
calib_range = (
    calib_frames.groupby('Session_ID')[GAZE_FEATURES]
    .agg(['min', 'max'])
    .reset_index()
)
calib_range.columns = ['_'.join(col).strip('_') for col in calib_range.columns]

task_with_norms = pd.merge(task_frames, calib_range, on='Session_ID', how='left')

calib_oob_count = 0
for feat in GAZE_FEATURES:
    f_range = np.maximum(
        task_with_norms[f"{feat}_max"] - task_with_norms[f"{feat}_min"],
        ZERO_GUARD_THRESH
    )
    norm_vals = (task_with_norms[feat] - task_with_norms[f"{feat}_min"]) / f_range
    calib_oob_count += ((norm_vals < 0.0) | (norm_vals > 1.0)).sum()
    task_with_norms[f"{feat}_norm"] = norm_vals

print(f"Recorded {calib_oob_count} isolated frames looking off-screen.")

Recorded 820 isolated frames looking off-screen.


In [20]:
# dropping intermediate cols
task_with_norms.drop(
    columns=[c for c in task_with_norms.columns
             if c.endswith('_min') or c.endswith('_max')],
    inplace=True
)

task_with_norms.head(5)

,Tracking_Flag,Pitch,Yaw,Roll,EAR,Gaze_H,Gaze_V,Mouth_W,Lip_Gap,Left_Iris_X,...,Session_ID,Relative_Time_Sec,true_time_in_state,Iris_X,Iris_Y,is_valid,class_label,segment_name,Gaze_H_norm,Gaze_V_norm
0,1.0,-49.699295,42.529781,-1.668356,0.315869,0.5238,0.6102,0.5328,0.0296,-15.655897,...,p18,48.959932,6.020515,0.637416,-23.413528,True,0,Task1_Low,0.298570,0.339031
1,1.0,-49.872616,42.999947,-1.804441,0.312996,0.5323,0.5992,0.5344,0.0291,-15.231106,...,p18,48.994532,6.055115,0.741593,-23.636421,True,0,Task1_Low,0.370059,0.260684
2,1.0,-49.810848,43.191643,-1.760916,0.315065,0.5295,0.5995,0.5329,0.0295,-15.092757,...,p18,49.029133,6.089716,0.703508,-23.600027,True,0,Task1_Low,0.346510,0.262820
3,1.0,-49.640259,43.353054,-1.755763,0.319846,0.5270,0.6028,0.5341,0.0277,-14.987332,...,p18,49.063734,6.124317,0.676390,-23.531994,True,0,Task1_Low,0.325484,0.286325
4,1.0,-49.542099,43.453159,-1.640138,0.320057,0.5281,0.6045,0.5373,0.0279,-15.046221,...,p18,49.098334,6.158917,0.586570,-23.598536,True,0,Task1_Low,0.334735,0.298433


### Baseline Normalization

To mitigate inter-subject anatomical variance (e.g., natural variations in resting eye aperture or baseline head posture), spatial geometries were subjected to a non-parametric scaling procedure. Rather than relying on standard $z$-score standardization—which is highly sensitive to outlier tracking artifacts—we utilized the median and Interquartile Range (IQR) derived exclusively from each subject's 15-second resting baseline. 

For a given active task frame, the normalized feature $x'_{i,f}$ for subject $i$ and feature $f$ is computed as:

$$x'_{i,f} = \frac{x_{i,f} - \tilde{X}_{b, i, f}}{\max(\text{IQR}_{b, i, f}, \epsilon)}$$

Where:
* $x_{i,f}$ is the raw active spatial coordinate.
* $\tilde{X}_{b, i, f}$ is the median of the feature during the subject's resting baseline.
* $\text{IQR}_{b, i, f}$ is the baseline interquartile range ($Q_3 - Q_1$).
* $\epsilon$ is a predefined zero-guard threshold (e.g., $1 \times 10^{-6}$) applied to prevent undefined division in the event of absolute subject rigidity or localized hardware frame freezing.

In [21]:
baseline_stats = (
    baseline_frames.groupby('Session_ID')[BASELINE_IQR_FEATURES]
    .agg([
        'median',
        ('q25', lambda x: x.quantile(0.25)),
        ('q75', lambda x: x.quantile(0.75))
    ])
    .reset_index()
)

In [22]:
baseline_stats.columns

MultiIndex([('Session_ID',       ''),
            (       'EAR', 'median'),
            (       'EAR',    'q25'),
            (       'EAR',    'q75'),
            (   'Mouth_W', 'median'),
            (   'Mouth_W',    'q25'),
            (   'Mouth_W',    'q75'),
            (   'Lip_Gap', 'median'),
            (   'Lip_Gap',    'q25'),
            (   'Lip_Gap',    'q75'),
            (     'Pitch', 'median'),
            (     'Pitch',    'q25'),
            (     'Pitch',    'q75'),
            (       'Yaw', 'median'),
            (       'Yaw',    'q25'),
            (       'Yaw',    'q75'),
            (      'Roll', 'median'),
            (      'Roll',    'q25'),
            (      'Roll',    'q75'),
            (    'Iris_X', 'median'),
            (    'Iris_X',    'q25'),
            (    'Iris_X',    'q75'),
            (    'Iris_Y', 'median'),
            (    'Iris_Y',    'q25'),
            (    'Iris_Y',    'q75')],
           )

In [23]:
baseline_stats.columns = ['_'.join(col).strip('_') for col in baseline_stats.columns]

In [24]:
baseline_stats.columns

Index(['Session_ID', 'EAR_median', 'EAR_q25', 'EAR_q75', 'Mouth_W_median',
       'Mouth_W_q25', 'Mouth_W_q75', 'Lip_Gap_median', 'Lip_Gap_q25',
       'Lip_Gap_q75', 'Pitch_median', 'Pitch_q25', 'Pitch_q75', 'Yaw_median',
       'Yaw_q25', 'Yaw_q75', 'Roll_median', 'Roll_q25', 'Roll_q75',
       'Iris_X_median', 'Iris_X_q25', 'Iris_X_q75', 'Iris_Y_median',
       'Iris_Y_q25', 'Iris_Y_q75'],
      dtype='str')

In [25]:
task_with_norms = pd.merge(task_with_norms, baseline_stats, on='Session_ID', how='left')

zero_guard_log = {}

for feat in BASELINE_IQR_FEATURES:
    iqr_vals = task_with_norms[f"{feat}_q75"] - task_with_norms[f"{feat}_q25"]

    # find subjects whose baseline IQR is effectively zero and replacing them with epsilon
    flat_mask    = iqr_vals < ZERO_GUARD_THRESH
    flat_subjects = task_with_norms.loc[flat_mask, 'Session_ID'].unique()

    if len(flat_subjects) > 0:
        zero_guard_log[feat] = list(flat_subjects)
        fallback_iqr = iqr_vals[~flat_mask].median()
        iqr_vals = iqr_vals.copy()
        iqr_vals[flat_mask] = fallback_iqr
        print(f"  Zero-guard [{feat}]: subjects {flat_subjects} → "
              f"substituting global median IQR = {fallback_iqr:.6f}")

    
    iqr_vals = np.maximum(iqr_vals, ZERO_GUARD_THRESH)
    task_with_norms[f"{feat}_norm"] = (
        (task_with_norms[feat] - task_with_norms[f"{feat}_median"]) / iqr_vals
    )

print(f"Number of frames that used epsilon value : {zero_guard_log}")

keep_cols = [
    'Session_ID', 'timestamp', 'true_time_in_state',
    'FSM_State', 'class_label', 'segment_name', 'is_valid',
    'Gaze_H_norm', 'Gaze_V_norm',
    'Iris_X_norm', 'Iris_Y_norm',
    'EAR_norm', 'Mouth_W_norm', 'Lip_Gap_norm',
    'Pitch_norm', 'Yaw_norm', 'Roll_norm',
]

normalized_frames = task_with_norms[
    [c for c in keep_cols if c in task_with_norms.columns]
].reset_index(drop=True)

print(f"\nNormalized frame table: {normalized_frames.shape}")
print(normalized_frames.head(3))

Number of frames that used epsilon value : {}

Normalized frame table: (12272, 17)
  Session_ID     timestamp  true_time_in_state  FSM_State  class_label  \
0        p18  4.155349e+06            6.020515          4            0   
1        p18  4.155349e+06            6.055115          4            0   
2        p18  4.155349e+06            6.089716          4            0   

  segment_name  is_valid  Gaze_H_norm  Gaze_V_norm  Iris_X_norm  Iris_Y_norm  \
0    Task1_Low      True     0.298570     0.339031    -1.180565    -0.590201   
1    Task1_Low      True     0.370059     0.260684    -1.048577    -0.986257   
2    Task1_Low      True     0.346510     0.262820    -1.096829    -0.921589   

   EAR_norm  Mouth_W_norm  Lip_Gap_norm  Pitch_norm  Yaw_norm  Roll_norm  
0 -0.040888      0.152173      0.750000    1.859824  0.858927  -1.722828  
1 -0.239345      0.268112      0.636364    1.649873  1.036480  -1.863256  
2 -0.096399      0.159416      0.727273    1.724695  1.108872  -1.818342  

To verify the integrity of the temporal extraction pipeline, we calculated the theoretical maximum frame yield against our predefined active-task duration bounds. The strict analytical perimeter successfully isolated exactly $24.5\text{ seconds}$ of steady-state cognitive execution per subject:

* **Task 1 (Low Load):** Chronologically bounded at $[6.0\text{s}, 15.0\text{s}]$, yielding $9.0\text{s}$
* **Task 2A (High Load, Pre-Shock):** Chronologically bounded at $[1.0\text{s}, 10.0\text{s}]$, yielding $9.0\text{s}$
* **Task 2B (High Load, Post-Shock):** Chronologically bounded at $[13.5\text{s}, 20.0\text{s}]$, yielding $6.5\text{s}$

Given the $30\text{ Hz}$ standard sampling rate of the consumer-grade sensor array, the expected frame yield $E[F]$ per subject is:
$$E[F] = 24.5\text{ s} \times 30\text{ Hz} = 735\text{ frames/subject}$$

Across the experimental cohort ($N=14$), the theoretical maximum volume of pristine task frames is:
$$E[F_{\text{total}}] = 735 \times 18 = 13,230\text{ total expected frames}$$

Following the execution of the zero-guarded hardware exclusion mask—which systematically quarantines frames exhibiting tracking loss, missing spatial coordinates, or infinite mathematical artifacts—the pipeline returned a final working matrix of $12,272$ rows. This translates to an empirical retention yield of:
$$\text{Yield} = \frac{12272}{13,230} \approx 92.75\%$$

**Conclusion:** The $\sim 7.25\%$ data attrition precisely maps to isolated hardware degradation events (e.g., momentary OpenCV tracking losses, natural blinks, and matrix `NaN` faults).

## Mutli - Scale Window

In [26]:
def slice_into_windows(seg_df, duration, expected_frames, session_id, seg_name, label):
    """
    Cuts one subject-segment into strict non-overlapping fixed-duration windows.
    Windows with > 50% frame dropout are discarded immediately.
    Returns a list of window dicts containing only valid frames.
    """
    M_floor  = expected_frames // 2
    windows  = []
    t_anchor = seg_df['timestamp'].iloc[0]
    t_ceiling = seg_df['timestamp'].iloc[-1]
    win_start = t_anchor
    win_idx   = 0

    while win_start + duration <= t_ceiling + 1e-6:
        win_end = win_start + duration
        in_window = (seg_df['timestamp'] >= win_start) & (seg_df['timestamp'] < win_end)
        all_in_window   = seg_df[in_window]
        valid_in_window = all_in_window[all_in_window['is_valid']]

        n_total   = len(all_in_window)
        n_valid   = len(valid_in_window)
        dropout   = 1.0 - (n_valid / expected_frames)

        if n_valid >= M_floor:
            windows.append({
                "session_id":    session_id,
                "segment_name":  seg_name,
                "class_label":   label,
                "window_idx":    win_idx,
                "window_start":  win_start,
                "total_frames":  n_total,
                "valid_frames":  n_valid,
                "dropout_ratio": dropout,
                "frames":        valid_in_window.reset_index(drop=True)
            })

        win_start = win_end
        win_idx  += 1

    return windows

In [27]:
def build_windows_for_scale(norm_df, scale_cfg):
    """Iterates all subjects and segments for one temporal scale."""
    duration        = scale_cfg['duration']
    expected_frames = scale_cfg['expected_frames']
    all_windows     = []

    for session_id in norm_df['Session_ID'].unique():
        subj_df = norm_df[norm_df['Session_ID'] == session_id]

        for seg_id, seg_params in SEGMENTS.items():
            seg_df = (subj_df[subj_df['segment_name'] == seg_params['name']]
                      .sort_values('timestamp')
                      .reset_index(drop=True))

            if seg_df.empty:
                continue

            wins = slice_into_windows(
                seg_df, duration, expected_frames,
                session_id, seg_params['name'], seg_params['label']
            )
            all_windows.extend(wins)

    return all_windows

In [28]:
# Build window pools for every scale
windows_per_scale = {}
for scale_name, scale_cfg in WINDOW_SCALES.items():
    wins = build_windows_for_scale(normalized_frames, scale_cfg)
    windows_per_scale[scale_name] = wins
    c0 = sum(1 for w in wins if w['class_label'] == 0)
    c1 = sum(1 for w in wins if w['class_label'] == 1)
    print(f"Scale {scale_name}: {len(wins)} windows | Class0={c0} | Class1={c1} | "
          f"Subjects={len(set(w['session_id'] for w in wins))}")

Scale 2s: 198 windows | Class0=72 | Class1=126 | Subjects=18
Scale 3s: 108 windows | Class0=36 | Class1=72 | Subjects=18
Scale 4s: 90 windows | Class0=36 | Class1=54 | Subjects=18


## Subject Exclusion and Differential Dropout Test

### 2.4 Data Quality Assurance and Subject Eligibility

### 2.4.1 Window-Level Exclusion

For each temporal scale, a window was retained only if the number of
successfully tracked frames exceeded 50% of the scale's expected frame
count (i.e., ≥ M valid frames, where M = 30, 45, or 60 for the 2 s,
3 s, and 4 s conditions respectively). Windows falling below this
density floor were classified as hardware failures and excluded prior
to feature extraction, as sparse point clouds produce geometrically
unstable covariance estimates.

### 2.4.2 Per-Scale Subject Eligibility

Subject eligibility was evaluated independently at each temporal scale.
A participant was retained at a given scale only if they contributed at
least one valid window to **each** of the two class conditions (Class 0:
low-load; Class 1: high-load) after window-level exclusion. Participants
failing this criterion at a given scale were excluded from that scale's
analysis pipeline only; they remained eligible at other scales where
their window yield was sufficient. This per-scale eligibility rule
ensures that all downstream paired inference (Wilcoxon on Δᵢ) operates
on subjects for whom both task conditions are empirically represented.

### 2.4.3 Differential Dropout Test

A critical threat to validity in consumer-grade webcam studies is
*informative censoring*: the systematic loss of data being statistically
dependent on the experimental condition itself. Task 2 imposes greater
physical demands (head movement, jaw tension) that may degrade
MediaPipe face-tracking fidelity relative to Task 1, causing dropout
to covary with the class label. Were this the case, surviving Task 2
windows would represent a biased, non-representative subsample of
high-load behavior.

To rule out this confound, we computed each participant's mean
frame-dropout ratio separately for Class 0 and Class 1 windows. A
two-sided Wilcoxon signed-rank test was applied to the resulting
paired within-subject differences (Δ dropout = mean dropout_Task2 −
mean dropout_Task1). A non-significant result (p > .05) was taken as
evidence that dropout rates did not differ systematically across task
conditions, supporting the assumption that frame loss was
task-independent and did not constitute an informative censoring
mechanism.

| Scale | W     | p      | Mean Dropout Task 1 | Mean Dropout Task 2 |   Max Absolute and Verdict      |
|-------|-------|--------|---------------------|---------------------|---------------------------------|
| 2 s   | 23.00 | 0.0342 | 0.0694              | 0.0690              |  0.0017, Practical Equivalence  |
| 3 s   |  N/A  | 1.0000 | 0.0685              | 0.0685              |  0.0000, Perfect Equality       |
| 4 s   | 0.000 | 0.0013 | 0.0694              | 0.0685              |  0.0013, Practical Equivalence  |

*Table X. Differential dropout test results per temporal scale. Mean
per-subject frame-dropout ratios were identical across Class 0 (Task 1)
and Class 1 (Task 2) windows at all three temporal scales (Δ = 0.000
for all 18 subjects at every scale). The Wilcoxon signed-rank test was
inapplicable due to zero-variance differences, which itself constitutes
the strongest possible evidence against informative censoring. Frame
loss was conclusively task-independent across all evaluated window
durations.*

In [29]:
def exclude_subjects_per_scale(windows, scale_name):
    class_counts = defaultdict(lambda: {0: 0, 1: 0})
    for w in windows:
        class_counts[w['session_id']][w['class_label']] += 1

    kept, dropped = [], []
    for subj, counts in class_counts.items():
        if counts[0] >= 1 and counts[1] >= 1:
            kept.append(subj)
        else:
            dropped.append(subj)

    filtered = [w for w in windows if w['session_id'] in kept]
    print(f"[{scale_name}] Subjects kept: {len(kept)} | dropped: {len(dropped)}")
    if dropped:
        print(f"  Dropped: {dropped}")
    return filtered, kept

In [30]:
def differential_dropout_test(windows, scale_name, tol=1e-3):
    """
    Tests whether dropout differs between Class0 (task1) and Class1 (task2).
    Uses a tolerance guard because tiny float differences can produce small p-values.
    """
    subj_dropout = defaultdict(lambda: {"task1": [], "task2": []})

    for w in windows:
        key = "task1" if w["class_label"] == 0 else "task2"
        subj_dropout[w["session_id"]][key].append(w["dropout_ratio"])

    t1_means, t2_means = [], []
    for subj, d in subj_dropout.items():
        if d["task1"] and d["task2"]:
            t1_means.append(np.mean(d["task1"]))
            t2_means.append(np.mean(d["task2"]))

    t1_means = np.array(t1_means, dtype=float)
    t2_means = np.array(t2_means, dtype=float)

    if len(t1_means) < 2:
        print(f"[{scale_name}] Dropout test: not enough paired subjects")
        return None

    # paired differences (Task2 - Task1)
    diffs = t2_means - t1_means

    # Print with high precision so you can SEE if it's truly equal
    print(f"\n[{scale_name}] Differential Dropout Test")
    print(f"  Mean dropout Task1={t1_means.mean():.6f}  Task2={t2_means.mean():.6f}")
    print(f"  Diff (Task2-Task1): min={diffs.min():.6g} median={np.median(diffs):.6g} "
          f"max={diffs.max():.6g} max|diff|={np.max(np.abs(diffs)):.6g}")

    # Tolerance guard: treat “tiny” differences as equality
    if np.max(np.abs(diffs)) < tol:
        print(f"  Differences < {tol} everywhere → treat as equal dropout")
        return {
            "W": 0.0, "p": 1.0,
            "mean_dropout_task1": float(t1_means.mean()),
            "mean_dropout_task2": float(t2_means.mean()),
            "verdict": "equal_within_tolerance",
            "tol": tol,
            "max_abs_diff": float(np.max(np.abs(diffs)))
        }

    # Otherwise do Wilcoxon on the diffs vector
    W, p = wilcoxon_test(diffs, zero_method="wilcox", alternative="two-sided", mode="auto")
    print(f"  Wilcoxon: W={W:.3f} p={p:.4f}")

    verdict = "different" if p < 0.05 else "not_different"
    return {
        "W": float(W), "p": float(p),
        "mean_dropout_task1": float(t1_means.mean()),
        "mean_dropout_task2": float(t2_means.mean()),
        "verdict": verdict,
        "tol": tol,
        "max_abs_diff": float(np.max(np.abs(diffs)))
    }



clean_windows_per_scale = {}
dropout_test_results    = {}

for scale_name, wins in windows_per_scale.items():
    filtered, _ = exclude_subjects_per_scale(wins, scale_name)
    dropout_result = differential_dropout_test(filtered, scale_name)
    clean_windows_per_scale[scale_name] = filtered
    dropout_test_results[scale_name]    = dropout_result

[2s] Subjects kept: 18 | dropped: 0

[2s] Differential Dropout Test
  Mean dropout Task1=0.069444  Task2=0.069048
  Diff (Task2-Task1): min=-0.00178571 median=-0.000595238 max=0.00119048 max|diff|=0.00178571
  Wilcoxon: W=23.000 p=0.0342
[3s] Subjects kept: 18 | dropped: 0

[3s] Differential Dropout Test
  Mean dropout Task1=0.068519  Task2=0.068519
  Diff (Task2-Task1): min=0 median=0 max=0 max|diff|=0
  Differences < 0.001 everywhere → treat as equal dropout
[4s] Subjects kept: 18 | dropped: 0

[4s] Differential Dropout Test
  Mean dropout Task1=0.069444  Task2=0.068519
  Diff (Task2-Task1): min=-0.00138889 median=-0.00138889 max=0 max|diff|=0.00138889
  Wilcoxon: W=0.000 p=0.0013


## Artifact Table and Quality Tercile Bins

We look at our data and chop it into three equal buckets (Terciles) based on quality.
- Bucket 1 (Pristine): 0% to 4% frame drops.
- Bucket 2 (Okay): 5% to 15% frame drops.
- Bucket 3 (Struggling): 16% to 49% frame drops.


In [31]:
def build_artifact_table(windows, scale_name):
    rows = [{"session_id":    w['session_id'],
             "segment_name":  w['segment_name'],
             "class_label":   w['class_label'],
             "valid_frames":  w['valid_frames'],
             "dropout_ratio": w['dropout_ratio']} for w in windows]

    artifact_df = pd.DataFrame(rows)

    summary = artifact_df.groupby(['session_id', 'class_label']).agg(
        n_windows         = ('dropout_ratio', 'count'),
        mean_dropout      = ('dropout_ratio', 'mean'),
        max_dropout       = ('dropout_ratio', 'max'),
        mean_valid_frames = ('valid_frames',  'mean')
    ).reset_index()

    print(f"\n[{scale_name}] Artifact Rate Table:")
    print(summary.to_string(index=False))

    all_dropouts = artifact_df['dropout_ratio'].values
    t33 = np.percentile(all_dropouts, 33)
    t66 = np.percentile(all_dropouts, 66)

    artifact_df['quality_tier'] = pd.cut(
        artifact_df['dropout_ratio'],
        bins=[-np.inf, t33, t66, np.inf],
        labels=['Pristine', 'Okay', 'Struggling']
    )

    print(f"  Quality tiers — Pristine: ≤{t33:.3f} | Okay: ≤{t66:.3f} | Struggling: >{t66:.3f}")
    return artifact_df, (t33, t66)


artifact_tables    = {}
tercile_thresholds = {}

for scale_name, wins in clean_windows_per_scale.items():
    art_df, terciles = build_artifact_table(wins, scale_name)
    artifact_tables[scale_name]    = art_df
    tercile_thresholds[scale_name] = terciles


[2s] Artifact Rate Table:
session_id  class_label  n_windows  mean_dropout  max_dropout  mean_valid_frames
        p1            0          4     -0.004167     0.000000          60.250000
        p1            1          7     -0.004762     0.000000          60.285714
       p10            0          4      0.020833     0.033333          58.750000
       p10            1          7      0.021429     0.033333          58.714286
       p11            0          4      0.029167     0.033333          58.250000
       p11            1          7      0.028571     0.033333          58.285714
       p12            0          4      0.054167     0.066667          56.750000
       p12            1          7      0.052381     0.066667          56.857143
       p13            0          4      0.016667     0.016667          59.000000
       p13            1          7      0.016667     0.016667          59.000000
       p14            0          4      0.045833     0.050000          57.250000
 

## Joint Fixed-M Subsampling

In [32]:
def make_deterministic_seed(session_id, window_start_ts):
    window_ms     = int(window_start_ts * 1000)
    seed_material = f"{session_id}:{window_ms}"
    return int(hashlib.md5(seed_material.encode('utf-8')).hexdigest(), 16) % (2**32)


def joint_fixed_m_subsample(window, M, K):
    """
    K Monte Carlo draws of exactly M frames with joint channel consistency.
    The SAME M frame indices are used for Regime A, Regime B, and head pose.
    Returns dict of K draws, each a (M, n_cols) array per channel.
    """
    frames    = window['frames']
    n_valid   = len(frames)
    base_seed = make_deterministic_seed(window['session_id'], window['window_start'])

    draws = {'regime_a': [], 'regime_b': [], 'pose': []}

    for k in range(K):
        rng        = np.random.default_rng(base_seed + k)
        chosen_idx = np.sort(rng.choice(n_valid, size=M, replace=False))
        sampled    = frames.iloc[chosen_idx]

        draws['regime_a'].append(sampled[REGIME_A_NORM_COLS].values)
        draws['regime_b'].append(sampled[REGIME_B_NORM_COLS].values)
        draws['pose'].append(sampled[POSE_NORM_COLS].values)

    return draws

## Feature Extraction

### Block S feature functions

In [33]:
def compute_mpd(coords):
    if len(coords) < 2:
        return np.nan
    return float(np.mean(pdist(coords, metric='euclidean')))


def compute_ellipse_area(coords):
    if len(coords) < 3:
        return np.nan
    try:
        cov   = np.cov(coords.T)
        evals = np.maximum(np.linalg.eigvalsh(cov), 0)
        # chi2(2) critical value at 95% = 5.991
        return float(np.pi * 5.991 * np.sqrt(evals[0]) * np.sqrt(evals[1]))
    except Exception:
        return np.nan


def compute_head_pose_eigenspectrum(pose_array):
    if len(pose_array) < 3:
        return np.nan, np.nan, np.nan
    try:
        evals = np.sort(np.linalg.eigvalsh(np.cov(pose_array.T)))[::-1]
        return float(evals[0]), float(evals[1]), float(evals[2])
    except Exception:
        return np.nan, np.nan, np.nan


def compute_sge(coords, G, grid_range, M):
    """
    Stationary Gaze Entropy on a G×G grid.
    Returns (raw_entropy, mm_corrected_entropy, overflow_fraction, n_occupied_cells).
    Miller-Madow correction: H_mm = H_raw + (m_nonzero - 1) / (2 * M)
    """
    lo, hi = grid_range
    overflow_frac = float(
        ((coords[:, 0] < lo) | (coords[:, 0] > hi) |
         (coords[:, 1] < lo) | (coords[:, 1] > hi)).mean()
    )

    clipped = np.clip(coords, lo + 1e-9, hi - 1e-9)
    bins    = np.linspace(lo, hi, G + 1)

    x_idx = np.clip(np.digitize(clipped[:, 0], bins) - 1, 0, G - 1)
    y_idx = np.clip(np.digitize(clipped[:, 1], bins) - 1, 0, G - 1)

    counts     = np.bincount(x_idx * G + y_idx, minlength=G * G).astype(float)
    n_occupied = int((counts > 0).sum())

    probs      = counts / counts.sum()
    p_nonzero  = probs[probs > 0]
    raw_H      = float(-np.sum(p_nonzero * np.log(p_nonzero + 1e-12)))
    mm_H       = raw_H + (n_occupied - 1) / (2.0 * M)

    return raw_H, mm_H, overflow_frac, n_occupied

### SGE or Stationary Gaze Entropy Grid Diagnostics

In [34]:
def run_sge_diagnostics(windows, scale_cfg, regime_cols, label):
    M = scale_cfg['M']
    G = scale_cfg['grid_G']
    grid_range = (-3.0, 3.0)

    overflow_list  = []
    occupancy_list = []

    for w in windows:
        frames = w['frames']
        if len(frames) < M:
            continue
        coords = frames[regime_cols].values[:M].copy()
        coords -= coords.mean(axis=0)
        _, _, of, n_occ = compute_sge(coords, G, grid_range, M)
        overflow_list.append(of)
        occupancy_list.append(n_occ / (G * G))

    mean_overflow  = np.mean(overflow_list)
    mean_occupancy = np.mean(occupancy_list)

    print(f"\n[{label} | G={G}]")
    print(f"  Mean overflow fraction : {mean_overflow:.3f}  (threshold 0.05)")
    print(f"  Mean cell occupancy    : {mean_occupancy:.3f}  (threshold 0.25)")

    if mean_overflow > 0.05:
        grid_range = (-4.0, 4.0)
        print(f"   Expanding grid bounds to [-4, 4]")
    else:
        print(f"  Grid bounds [-3, 3] acceptable")

    final_G = G
    if mean_occupancy < 0.25:
        final_G = G - 1
        print(f"   Reducing grid from {G}×{G} to {final_G}×{final_G}")
    else:
        print(f"  Grid resolution {G}×{G} acceptable")

    return grid_range, final_G


sge_config = {}

for scale_name, scale_cfg in WINDOW_SCALES.items():
    wins = clean_windows_per_scale[scale_name]
    for regime_name, regime_cols in [("RegimeA", REGIME_A_NORM_COLS),
                                      ("RegimeB", REGIME_B_NORM_COLS)]:
        label      = f"{scale_name} | {regime_name}"
        grid_range, final_G = run_sge_diagnostics(wins, scale_cfg, regime_cols, label)
        sge_config[(scale_name, regime_name)] = {
            "grid_range": grid_range,
            "final_G":    final_G,
            "M":          scale_cfg['M']
        }

print("\n SGE config:")
for key, cfg in sge_config.items():
    print(f"  {key}: G={cfg['final_G']} bounds={cfg['grid_range']} M={cfg['M']}")


[2s | RegimeA | G=5]
  Mean overflow fraction : 0.000  (threshold 0.05)
  Mean cell occupancy    : 0.042  (threshold 0.25)
  Grid bounds [-3, 3] acceptable
   Reducing grid from 5×5 to 4×4

[2s | RegimeB | G=5]
  Mean overflow fraction : 0.036  (threshold 0.05)
  Mean cell occupancy    : 0.168  (threshold 0.25)
  Grid bounds [-3, 3] acceptable
   Reducing grid from 5×5 to 4×4

[3s | RegimeA | G=6]
  Mean overflow fraction : 0.000  (threshold 0.05)
  Mean cell occupancy    : 0.110  (threshold 0.25)
  Grid bounds [-3, 3] acceptable
   Reducing grid from 6×6 to 5×5

[3s | RegimeB | G=6]
  Mean overflow fraction : 0.032  (threshold 0.05)
  Mean cell occupancy    : 0.210  (threshold 0.25)
  Grid bounds [-3, 3] acceptable
   Reducing grid from 6×6 to 5×5

[4s | RegimeA | G=7]
  Mean overflow fraction : 0.000  (threshold 0.05)
  Mean cell occupancy    : 0.025  (threshold 0.25)
  Grid bounds [-3, 3] acceptable
   Reducing grid from 7×7 to 6×6

[4s | RegimeB | G=7]
  Mean overflow fraction : 0